# Alphonsus de Liguori Qwen3 14B Fine-Tuning with Unsloth (4-bit QLoRA)

**Base Model:** Qwen3 14B | **Dataset:** Liguori ShareGPT JSONL | **Template:** ChatML

Single-persona LoRA — Saint Alphonsus de Liguori, devotional voice mode.

## 1. Configuration

In [1]:
PROJECT_ROOT = "/home/spark/projects/training/biblical"
DATA_DIR = f"{PROJECT_ROOT}/data"
OUTPUT_ROOT = f"{PROJECT_ROOT}/output"

BASE_LLM = "unsloth/Qwen3-14B-unsloth-bnb-4bit"
MODEL_NAME_BASE = "liguori_qwen3_14b_unsloth_4bit"

INPUT_DATA_FILE = f"{DATA_DIR}/training-data/liguori_persona/liguori_sharegpt.jsonl"

VOICE_MODES = {
    "devotional": {
        "detection_phrase": "Saint Alphonsus Maria de Liguori",
        "description": "Ardent devotional prayer, pastoral counsel, mystical longing",
        "test_prompt": "How should I pray when I feel nothing and God seems far away?",
    },
}

OUTPUT_BASE_DIR = f"{OUTPUT_ROOT}/{MODEL_NAME_BASE}"
OUTPUT_DIR_ADAPTERS = f"{OUTPUT_BASE_DIR}/train"
LORA_OUTPUT_DIR = f"{OUTPUT_BASE_DIR}/lora_adapters"

MAX_SEQ_LENGTH = 4096
BATCH_SIZE = 2
GRAD_ACCUM = 4
LEARNING_RATE = 1e-4
TARGET_EPOCHS = 1

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]

TEST_PROMPT = "What do you say to the soul that fears it has sinned too greatly to be forgiven?"

print("✓ Configuration loaded (Liguori Qwen3 14B 4-bit QLoRA)")
print(f"  Base model: {BASE_LLM}")
print(f"  Input data: {INPUT_DATA_FILE}")
print(f"  LoRA output: {LORA_OUTPUT_DIR}")
print(f"  Max seq length: {MAX_SEQ_LENGTH}")

✓ Configuration loaded (Liguori Qwen3 14B 4-bit QLoRA)
  Base model: unsloth/Qwen3-14B-unsloth-bnb-4bit
  Input data: /home/spark/projects/training/biblical/data/training-data/liguori_persona/liguori_sharegpt.jsonl
  LoRA output: /home/spark/projects/training/biblical/output/liguori_qwen3_14b_unsloth_4bit/lora_adapters
  Max seq length: 4096


## 2. Environment Preparation

In [2]:
!pip install -q unsloth transformers trl peft accelerate datasets bitsandbytes

import unsloth
import transformers
import trl
print(f"✓ Unsloth: {unsloth.__version__}")
print(f"✓ Transformers: {transformers.__version__}")
print(f"✓ TRL: {trl.__version__}")
print("Environment ready!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/spark/projects/quantization/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/spark/projects/quantization/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:435: UserWarning: 
    Found GPU0 NVIDIA GB10 which is of cuda capability 12.1.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (8.0) - (12.0)
    
  queued_call()


🦥 Unsloth Zoo will now patch everything to make training faster!
✓ Unsloth: 2026.2.1
✓ Transformers: 4.57.3
✓ TRL: 0.24.0
Environment ready!


## 3. Load Dataset

Extract the system prompt from the JSONL at load time.

In [3]:
import json, os
from collections import defaultdict
from datasets import Dataset as HFDataset

print(f"LOADING LIGUORI SHAREGPT DATA")
print(f"  File: {INPUT_DATA_FILE}")

conversations = []
voice_mode_system_prompts = {}
mode_counts = defaultdict(int)

with open(INPUT_DATA_FILE) as f:
    for line in f:
        conv = json.loads(line)
        conversations.append(conv)
        sys_msg = conv["conversations"][0]["value"]
        detected = "unknown"
        for mode_name, mode_info in VOICE_MODES.items():
            if mode_info["detection_phrase"] in sys_msg:
                detected = mode_name
                if mode_name not in voice_mode_system_prompts:
                    voice_mode_system_prompts[mode_name] = sys_msg
                break
        mode_counts[detected] += 1

dataset = HFDataset.from_list(conversations)

print(f"\n{'='*50}")
print(f"Total dataset: {len(dataset)} conversations")
print(f"Extracted {len(voice_mode_system_prompts)} voice mode system prompts")
for m, c in sorted(mode_counts.items(), key=lambda x: -x[1]):
    print(f"  {m:20s} {c:>5d} conversations")

if voice_mode_system_prompts:
    sample_mode = next(iter(voice_mode_system_prompts))
    print(f"\n--- Sample extracted prompt ({sample_mode}, first 250 chars) ---")
    print(f"  {voice_mode_system_prompts[sample_mode][:250]}...")

LOADING LIGUORI SHAREGPT DATA
  File: /home/spark/projects/training/biblical/data/training-data/liguori_persona/liguori_sharegpt.jsonl

Total dataset: 844 conversations
Extracted 1 voice mode system prompts
  devotional             844 conversations

--- Sample extracted prompt (devotional, first 250 chars) ---
  You are Saint Alphonsus Maria de Liguori, Doctor of the Church, founder of the Congregation of the Most Holy Redeemer (Redemptorists), a moral theologian who wrote with burning devotion about the love of God, the Passion of Christ, and total conformi...


## 4. Validate Dataset

In [4]:

bad_examples = []
empty_responses = []
unique_system_prompts = set()

for i, example in enumerate(dataset):
    convs = example["conversations"]
    # Multi-turn ShareGPT: system, then alternating human/gpt pairs
    if len(convs) < 3 or len(convs) % 2 == 0:
        bad_examples.append((i, f"Expected odd turn count ≥3, got {len(convs)}"))
        continue
    if convs[0]["from"] != "system":
        bad_examples.append((i, f"First turn should be 'system', got '{convs[0]['from']}'"))
        continue
    # Validate alternating human/gpt after system
    role_ok = True
    for j in range(1, len(convs)):
        expected = "human" if j % 2 == 1 else "gpt"
        if convs[j]["from"] != expected:
            bad_examples.append((i, f"Turn {j} should be '{expected}', got '{convs[j]['from']}'"))
            role_ok = False
            break
    if not role_ok:
        continue
    # Check last GPT response is not empty
    if len(convs[-1]["value"].strip()) == 0:
        empty_responses.append(i)
    unique_system_prompts.add(convs[0]["value"])

# Turn-count distribution
from collections import Counter
turn_dist = Counter(len(ex["conversations"]) for ex in dataset)

print("DATA QUALITY CHECK")
print(f"  Total examples: {len(dataset)}")
print(f"  Bad structure: {len(bad_examples)}")
print(f"  Empty responses: {len(empty_responses)}")
print(f"  Unique system prompts: {len(unique_system_prompts)} (expected: ~{len(voice_mode_system_prompts)} voice modes)")
print(f"  Turn distribution: {dict(sorted(turn_dist.items()))}")

if bad_examples:
    print(f"\n⚠️ Bad examples (first 5):")
    for idx, reason in bad_examples[:5]:
        print(f"    Example {idx}: {reason}")

if empty_responses:
    print(f"\n⚠️ Filtering {len(empty_responses)} empty responses...")
    good_indices = [i for i in range(len(dataset)) if i not in set(empty_responses)]
    dataset = dataset.select(good_indices)
    print(f"  Dataset after filtering: {len(dataset)} examples")

# Voice mode distribution with balance check
print(f"\nVOICE MODE DISTRIBUTION:")
for mode, count in sorted(mode_counts.items(), key=lambda x: -x[1]):
    bar = "█" * (count // 20) + "▌" * (1 if count % 20 >= 10 else 0)
    print(f"  {mode:<16} {count:>5}  {bar}")
print(f"  {'TOTAL':<16} {sum(mode_counts.values()):>5}")

# Class imbalance check
known_counts = [c for m, c in mode_counts.items() if m != "unknown"]
if known_counts:
    smallest, largest = min(known_counts), max(known_counts)
    if smallest < largest * 0.15:
        print(f"\n⚠️ CLASS IMBALANCE: smallest mode ({smallest}) is <15% of largest ({largest})")
        print("  Consider upsampling underrepresented modes or generating more data.")
    else:
        print(f"\n✓ Balance OK: smallest mode is {smallest/largest*100:.0f}% of largest")

# Show voice differentiation — one sample from each mode
print(f"\nVOICE SAMPLES (first ~120 chars of response per mode):")
seen_modes = set()
for example in dataset:
    sys_msg = example["conversations"][0]["value"]
    for mode_name, mode_info in VOICE_MODES.items():
        if mode_info["detection_phrase"] in sys_msg and mode_name not in seen_modes:
            response_start = example["conversations"][2]["value"][:120]
            print(f"  [{mode_name:14s}] \"{response_start}...\"")
            seen_modes.add(mode_name)
            break
    if len(seen_modes) >= len(VOICE_MODES):
        break

print(f"\n✓ Dataset validated and ready for training")


DATA QUALITY CHECK
  Total examples: 844
  Bad structure: 0
  Empty responses: 0
  Unique system prompts: 1 (expected: ~1 voice modes)
  Turn distribution: {5: 3, 7: 201, 9: 640}

VOICE MODE DISTRIBUTION:
  devotional         844  ██████████████████████████████████████████
  TOTAL              844

✓ Balance OK: smallest mode is 100% of largest

VOICE SAMPLES (first ~120 chars of response per mode):
  [devotional    ] "O my crucified Jesus, how many tears have streamed from the eyes of anxious souls who tremble at every shadow of sin, wh..."

✓ Dataset validated and ready for training


## 5. Load Model & Tokenizer (4-bit)

In [5]:
from unsloth import FastLanguageModel
import torch

# Load model in 4-bit precision for QLoRA training
# Use device_map={"":  0} to force all layers onto GPU 0
# DGX Spark has 119.7GB \u2014 plenty for 14B 4-bit (~8GB)
# device_map="auto" incorrectly offloads some layers to CPU which BnB 4-bit rejects
model, tokenizer = FastLanguageModel.from_pretrained(
    BASE_LLM,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,           # Auto-detect (will use bfloat16 for compute)
    load_in_4bit=True,    # 4-bit quantization for QLoRA
    device_map={"":  0}    # Force all layers onto GPU 0 (119GB unified memory)
)

# Only set pad_token if not already defined by the model
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"\u2713 Model loaded: {BASE_LLM}")
print(f"\u2713 Precision: 4-bit (QLoRA)")

print(f"\u2713 Pad token: {repr(tokenizer.pad_token)}")
print(f"\u2713 Max sequence length: {MAX_SEQ_LENGTH}")

==((====))==  Unsloth 2026.2.1: Fast Qwen3 patching. Transformers: 4.57.3.
   \\   /|    NVIDIA GB10. Num GPUs = 1. Max memory: 121.69 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu130. CUDA: 12.1. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


KeyboardInterrupt: 

## 6. Format Dataset for Chat Template

In [ ]:
# Format dataset for Qwen3 chat template (ChatML) using Unsloth's standardize_sharegpt
from unsloth.chat_templates import standardize_sharegpt

dataset = standardize_sharegpt(dataset)
formatted_texts = tokenizer.apply_chat_template(
    list(dataset["conversations"]),
    tokenize=False,
)

# Build final dataset
import pandas as pd
from datasets import Dataset as HFDataset
dataset = HFDataset.from_pandas(pd.DataFrame({"text": formatted_texts}))

# Filter out empty examples and shuffle
dataset = dataset.filter(lambda x: len(x["text"]) > 0)
dataset = dataset.shuffle(seed=42)

print(f"--- Sample formatted text (first 500 chars) ---")
print(dataset[0]['text'][:500])
print(f"\n\u2713 Dataset formatted: {len(dataset)} examples")

## 7. Add LoRA Adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=LORA_TARGET_MODULES,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    max_seq_length=MAX_SEQ_LENGTH
)

print(f"\u2713 LoRA adapters added (r={LORA_R}, targets={LORA_TARGET_MODULES})")
print(f"\u2713 Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 8. Trainer Setup

In [ ]:
from trl import SFTTrainer, SFTConfig

# Calculate training steps
effective_batch_size = BATCH_SIZE * GRAD_ACCUM
steps_per_epoch = len(dataset) // effective_batch_size
max_steps = steps_per_epoch * TARGET_EPOCHS
warmup_steps = max(1, max_steps // 10)
save_steps = min(100, steps_per_epoch)  # Save every 100 steps or at epoch end

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LENGTH,
        packing=False,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_steps=warmup_steps,
        max_steps=max_steps,
        learning_rate=LEARNING_RATE,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir=OUTPUT_DIR_ADAPTERS,
        report_to="none",
        save_strategy="steps",
        save_steps=save_steps,
    ),
)

print("✓ Trainer configured for Augustine 14B 4-bit QLoRA training")
print(f"✓ Dataset size: {len(dataset)} multi-turn conversations")
print(f"✓ Effective batch size: {effective_batch_size} (batch={BATCH_SIZE} × grad_accum={GRAD_ACCUM})")
print(f"✓ Steps per epoch: {steps_per_epoch}")
print(f"✓ Total epochs: {TARGET_EPOCHS}")
print(f"✓ Total steps: {max_steps}")
print(f"✓ Warmup steps: {warmup_steps}")
print(f"✓ Save every: {save_steps} steps")
print(f"✓ Learning rate: {LEARNING_RATE}")

## 9. Train

In [ ]:
trainer.train()

## 10. Save LoRA Adapters

In [ ]:

from pathlib import Path
import json

Path(LORA_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# Save LoRA adapters + tokenizer
print(f"Saving LoRA adapters to {LORA_OUTPUT_DIR}...")
model.save_pretrained(LORA_OUTPUT_DIR)
tokenizer.save_pretrained(LORA_OUTPUT_DIR)

# Save voice mode system prompts alongside adapters for inference use
prompts_path = f"{LORA_OUTPUT_DIR}/voice_mode_system_prompts.json"
with open(prompts_path, "w") as f:
    json.dump(voice_mode_system_prompts, f, indent=2)

print(f"\n✓ LoRA adapters saved!")
print(f"  Adapters:       {LORA_OUTPUT_DIR}")
print(f"  System prompts: {prompts_path} ({len(voice_mode_system_prompts)} voice modes)")
print(f"\n  At inference, load prompts with:")
print(f'    with open("{prompts_path}") as f:')
print(f'        prompts = json.load(f)')
print(f'    system_msg = prompts["confessional"]  # or philosophical, polemical, devotional')


## 11. Test Inference

In [ ]:

FastLanguageModel.for_inference(model)

# Use the system prompts already extracted at load time (voice_mode_system_prompts)
for mode_name, mode_info in VOICE_MODES.items():
    system_prompt = voice_mode_system_prompts.get(mode_name, "You are Saint Augustine of Hippo.")
    test_prompt = mode_info["test_prompt"]

    test_messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": test_prompt},
    ]

    inputs = tokenizer.apply_chat_template(
        [test_messages],
        tokenize=True,
        return_tensors="pt",
        add_generation_prompt=True,
    ).to(model.device)

    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=256,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
    )

    response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)

    print(f"{'='*60}")
    print(f"  AUGUSTINE — {mode_name.upper()} MODE")
    print(f"  {mode_info['description']}")
    print(f"{'='*60}")
    print(f"Q: {test_prompt}")
    print(f"\nA: {response}")
    print()


## 12. Verify Saved Adapter

In [ ]:

print(f"Verifying saved adapter from {LORA_OUTPUT_DIR}...")

verify_model, verify_tokenizer = FastLanguageModel.from_pretrained(
    LORA_OUTPUT_DIR,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
    device_map={"": 0},
)

FastLanguageModel.for_inference(verify_model)

# Test with confessional mode — the most distinctive register
system_prompt = voice_mode_system_prompts.get("confessional", "You are Saint Augustine of Hippo.")
test_messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": TEST_PROMPT},
]

inputs = verify_tokenizer.apply_chat_template(
    [test_messages],
    tokenize=True,
    return_tensors="pt",
    add_generation_prompt=True,
).to(verify_model.device)

outputs = verify_model.generate(
    input_ids=inputs,
    max_new_tokens=128,
    temperature=0.7,
    do_sample=True,
)

response = verify_tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
print(f"\nAUGUSTINE (confessional): {response}")
print(f"\n✓ Adapter verified!")

del verify_model, verify_tokenizer
torch.cuda.empty_cache()
print("✓ Verification model cleared from memory")
